In [9]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [12]:
url = "https://www.nykaa.com/search/result/?q=baby&root=search&searchType=Manual&sourcepage=home"

In [13]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [14]:
response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)
print("Length:", len(response.content))

Status Code: 200
Length: 557779


In [16]:
soup = BeautifulSoup(response.content, "html.parser")

titles = soup.find_all("h2", class_="css-xrzmfa")

print("Number of products:", len(titles))

Number of products: 20


In [17]:
# Empty list to store product data
products = []

for product in titles[:20]:

    # Product Name
    product_name = product.get_text(strip=True)

    # Product card text
    parent = product.parent
    text = parent.get_text(" ", strip=True)

    # MRP
    mrp_match = re.search(r"Regular price ₹(\d+)", text)

    if mrp_match:
        mrp = mrp_match.group(1)
    else:
        mrp = "N/A"

    # Selling Price
    price_match = re.search(r"Discounted price ₹(\d+)", text)

    if price_match:
        price = price_match.group(1)
    else:
        price = "N/A"

    # Discount
    discount_match = re.search(r"(\d+)% Off", text)

    if discount_match:
        discount = discount_match.group(1) + "%"
    else:
        discount = "N/A"

    # Brand
    brand_match = re.match(r"(\S+)", product_name)

    if brand_match:
        brand = brand_match.group(1)
    else:
        brand = "N/A"

    # Rating
    rating = "N/A"

    current = product

    for i in range(5):

        if current:

            current_text = current.get_text(" ", strip=True)

            rating_match = re.search(r"\b[0-5]\.\d\b", current_text)

            if rating_match:
                rating = rating_match.group(0)
                break

            current = current.parent

    # Reviews
    reviews_match = re.search(r"\(\s*(\d+)\s*\)", text)

    if reviews_match:
        reviews = reviews_match.group(1)
    else:
        reviews = "N/A"

    # Product URL
    link = product.find_parent("a", href=True)

    if link:
        product_url = "https://www.nykaa.com" + link["href"]
    else:
        product_url = "N/A"

    # Category
    category = "Baby"

    # Store product data
    products.append({
        "Product Name": product_name,
        "Brand": brand,
        "MRP": mrp,
        "Selling Price": price,
        "Discount": discount,
        "Rating": rating,
        "Reviews": reviews,
        "Product URL": product_url,
        "Category": category
    })




In [18]:
df = pd.DataFrame(products)

df

,Product Name,Brand,MRP,Selling Price,Discount,Rating,Reviews,Product URL,Category
0,BABY FOREST Maasoom Maalish Baby Body Massage ...,BABY,N/A,N/A,N/A,N/A,15,https://www.nykaa.com/baby-forest-maasoom-maal...,Baby
1,"Cetaphil Baby Daily Lotion With Shea Butter, p...",Cetaphil,300,275,0%,N/A,13598,https://www.nykaa.com/cetaphil-baby-daily-loti...,Baby
2,Aveeno Baby Daily Moisture Wash & Shampoo - Na...,Aveeno,1379,1299,6%,N/A,5227,https://www.nykaa.com/aveeno-baby-daily-moistu...,Baby
3,Aveeno Baby Daily Moisture Lotion | Oatmeal Fa...,Aveeno,1050,949,10%,N/A,1806,https://www.nykaa.com/aveeno-baby-daily-moistu...,Baby
4,Old School Handmade By Grandmas Baby Ubtan Bat...,Old,950,874,8%,N/A,N/A,https://www.nykaa.com/old-school-handmade-by-g...,Baby
5,Aveeno Baby DM Cleansing Bar,Aveeno,240,190,21%,N/A,464,https://www.nykaa.com/aveeno-baby-dm-cleansing...,Baby
6,Cetaphil Baby Gentle Wash & Shampoo With 5 Fol...,Cetaphil,275,250,0%,N/A,5336,https://www.nykaa.com/cetaphil-baby-gentle-was...,Baby
7,"Cetaphil Baby Mild Bar For Sensitive Skin, Par...",Cetaphil,260,237,9%,N/A,3517,https://www.nykaa.com/cetaphil-baby-mild-bar/p...,Baby
8,BABY FOREST Dulaar Pyar Talc-Free Baby Powder ...,BABY,N/A,N/A,N/A,N/A,39,https://www.nykaa.com/baby-forest-dulaar-pyar-...,Baby
9,Baby Dove Rich Moisture Baby Wash Moisturizing...,Baby,400,340,15%,N/A,2799,https://www.nykaa.com/dove-rich-moisture-hypoa...,Baby


In [19]:
df.to_csv("nykaa_baby_products.csv", index=False)

print("CSV file created successfully!")
print("Total products saved:", len(df))

CSV file created successfully!
Total products saved: 20
